In [1]:
import sys
import torch
import math
from xray_gaussian_rasterization_voxelization import (
    GaussianRasterizationSettings,
    GaussianRasterizer,
    GaussianVoxelizationSettings,
    GaussianVoxelizer,
)

sys.path.append("./")
from r2_gaussian.gaussian.gaussian_model import GaussianModel
from r2_gaussian.dataset.cameras import Camera
from r2_gaussian.arguments import PipelineParams

In [2]:
from r2_gaussian.dataset import Scene
from r2_gaussian.gaussian import GaussianModel, render, query, initialize_gaussian

In [3]:
from argparse import ArgumentParser
from r2_gaussian.arguments import (
    ModelParams,
    PipelineParams,
    get_combined_args,
)

In [4]:
from argparse import Namespace

In [135]:
from argparse import ArgumentParser, Namespace
import torch

def load_vol_pred_from_model_path(
    model_path: str,
    source_path: str = "/home/maemaeko/imari_lab/r2_gaussian/data/real_dataset/cone_ntrain_75_angle_360/teapot",
    loaded_iter: int = 1000,
    data_device: str = "cuda",
    scale_min: float = 0.0005,
    scale_max: float = 0.5,
    eval_mode: bool = True,
    nVoxel=(256, 256, 256),
    vol_center=(0, 0, 0),
):
    """
    model_path を入力として、query() で得られる vol_pred (D,H,W) を返す。

    依存:
      - ModelParams, PipelineParams, Scene
      - GaussianModel, initialize_gaussian
      - query
    """
    # --- parser / params ---
    parser = ArgumentParser(description="Testing script parameters")
    model = ModelParams(parser, sentinel=True)
    pipeline = PipelineParams(parser)

    # --- args ---
    args = Namespace(
        source_path=source_path,
        model_path=model_path,
        data_device=data_device,
        scale_min=scale_min,
        scale_max=scale_max,
        eval=eval_mode,
    )

    # --- dataset / scene ---
    dataset = model.extract(args)
    scene = Scene(dataset, shuffle=False)

    # --- load gaussians ---
    gaussians = GaussianModel(None)  # scale_bound will be loaded later
    _ = initialize_gaussian(gaussians, dataset, loaded_iter=loaded_iter)
    scene.gaussians = gaussians

    # --- volume query params ---
    scanner_cfg = scene.scanner_cfg
    tv_vol_center = torch.tensor(vol_center, device=gaussians.get_xyz.device)
    tv_vol_nVoxel = torch.tensor(nVoxel, device=gaussians.get_xyz.device)
    tv_vol_sVoxel = torch.tensor(scanner_cfg["dVoxel"], device=gaussians.get_xyz.device) * tv_vol_nVoxel

    # --- query ---
    query_pkg = query(
        gaussians,
        tv_vol_center,
        tv_vol_nVoxel,
        tv_vol_sVoxel,
        pipeline,
    )
    vol_pred = query_pkg["vol"]  # expected (D,H,W) or similar

    return vol_pred


# 使い方例:
vol_pred = load_vol_pred_from_model_path(
    model_path="/home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_9_angle_360-wo-densification/0_jaw_cone",
    source_path="/home/maemaeko/imari_lab/r2_gaussian/data/real_dataset/cone_ntrain_75_angle_360/teapot",
    loaded_iter=1000,
)
print(vol_pred.shape, vol_pred.device, vol_pred.dtype)


Reading camera 75/75 for train
Reading camera 100/100 for test
Loading Training Cameras
Loading Test Cameras
Loading trained model at iteration 1000
Loading from /home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_9_angle_360-wo-densification/0_jaw_cone/point_cloud/iteration_1000/point_cloud.pickle
torch.Size([256, 256, 256]) cuda:0 torch.float32


In [136]:
# 使い方例:
vol_pred_compare = load_vol_pred_from_model_path(
    model_path="/home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_9_angle_360-random-sampling-loss-wo-densification-wo-filter/0_jaw_cone",
    source_path="/home/maemaeko/imari_lab/r2_gaussian/data/real_dataset/cone_ntrain_75_angle_360/teapot",
    loaded_iter=1000,
)
print(vol_pred_compare.shape, vol_pred_compare.device, vol_pred_compare.dtype)


Reading camera 75/75 for train
Reading camera 100/100 for test
Loading Training Cameras
Loading Test Cameras
Loading trained model at iteration 1000
Loading from /home/maemaeko/imari_lab/r2_gaussian/output/synthetic_dataset/cone_ntrain_9_angle_360-random-sampling-loss-wo-densification-wo-filter/0_jaw_cone/point_cloud/iteration_1000/point_cloud.pickle
torch.Size([256, 256, 256]) cuda:0 torch.float32


In [137]:
import os
import numpy as np
import torch
import imageio.v2 as imageio

def save_compare_leak_overlay(
    vol_base: torch.Tensor,          # vol_pred
    vol_compare: torch.Tensor,       # vol_pred_compare
    out_dir: str = "overlay_leak_frames",
    order: str = "DHW",              # "DHW" or "HWD"
    compare_thr: float = 0.3,        # compare >= 0.3 を表示
    base_eps: float = 1e-6,          # base ≈ 0 の許容
    bg_percentile=(1, 99),           # 背景（base）の正規化
    alpha: float = 0.55,             # 赤の重ね具合
    prefix: str = "overlay",
    ext: str = "png",
):
    """
    mask条件:
      (abs(base) <= base_eps) AND (compare >= compare_thr)
    を赤で重ねて、slice画像として保存
    """
    assert vol_base.shape == vol_compare.shape
    assert vol_base.ndim == 3

    a = vol_base.detach()
    b = vol_compare.detach()
    if a.is_cuda: a = a.cpu()
    if b.is_cuda: b = b.cpu()
    a = a.float().numpy()
    b = b.float().numpy()

    # (H,W,D) -> (D,H,W) に統一
    if order.upper() == "HWD":
        a = np.transpose(a, (2, 0, 1))
        b = np.transpose(b, (2, 0, 1))
    elif order.upper() != "DHW":
        raise ValueError('order must be "DHW" or "HWD"')

    # mask: base≈0 なのに compareが大きい
    print(b.max(), b.min())
    #mask = (np.abs(a) <= base_eps) & (b > 0.)
    #mask = np.abs(a) <= base_eps
    mask = np.abs(b-a) > 0.05

    # 背景（base）を全スライス共通で正規化
    lo, hi = np.percentile(a, bg_percentile)
    if hi <= lo:
        hi = lo + 1e-6

    os.makedirs(out_dir, exist_ok=True)

    for z in range(a.shape[0]):
        img = b[z]            # (H,W)
        mm  = mask[z]         # (H,W)

        img01 = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
        gray = (img01 * 255.0).astype(np.uint8)

        rgb = np.stack([gray, gray, gray], axis=-1).astype(np.float32)

        red = np.zeros_like(rgb)
        red[..., 0] = 255.0
        rgb[mm] = (1.0 - alpha) * rgb[mm] + alpha * red[mm]

        imageio.imwrite(os.path.join(out_dir, f"{prefix}_{z:04d}.{ext}"), rgb.astype(np.uint8))

    print(f"[save_compare_leak_overlay] compare_thr={compare_thr}, base_eps={base_eps}, "
          f"mask_count={mask.sum()} / {mask.size}, saved to: {out_dir}")

# 使い方:
save_compare_leak_overlay(vol_pred, vol_pred_compare,
                          out_dir="overlay_base0_compare0p3",
                          order="HWD",
                          compare_thr=0.00001,
                          base_eps=1e-1,
                          alpha=0.6)


1.4881692 3.5586865e-05
[save_compare_leak_overlay] compare_thr=1e-05, base_eps=0.1, mask_count=38773 / 16777216, saved to: overlay_base0_compare0p3


In [138]:
import os
import numpy as np
import torch
import imageio.v2 as imageio

def save_compare_ratio_overlay(
    vol_base: torch.Tensor,          # vol_pred
    vol_compare: torch.Tensor,       # vol_pred_compare
    out_dir: str = "overlay_ratio_frames",
    order: str = "DHW",              # "DHW" or "HWD"
    ratio_thr: float = 2.0,          # compare が base の何倍以上なら赤にするか
    base_eps: float = 1e-6,          # 比率の発散防止
    compare_min: float = 0.0,        # compareが小さいノイズを除外（例: 0.05 など）
    use_abs_base: bool = True,       # baseが負もあり得るなら True 推奨
    bg_percentile=(1, 99),           # 背景（base）の正規化
    alpha: float = 0.55,
    prefix: str = "overlay",
    ext: str = "png",
):
    """
    mask条件:
      ratio = compare / (|base| + eps)  (or compare/(base+eps))
      mask = (ratio >= ratio_thr) & (compare >= compare_min)
    """
    assert vol_base.shape == vol_compare.shape
    assert vol_base.ndim == 3

    a = vol_base.detach()
    b = vol_compare.detach()
    if a.is_cuda: a = a.cpu()
    if b.is_cuda: b = b.cpu()
    a = a.float().numpy()
    b = b.float().numpy()

    # (H,W,D) -> (D,H,W) に統一
    if order.upper() == "HWD":
        a = np.transpose(a, (2, 0, 1))
        b = np.transpose(b, (2, 0, 1))
    elif order.upper() != "DHW":
        raise ValueError('order must be "DHW" or "HWD"')

    denom = (np.abs(a) if use_abs_base else a) + base_eps
    ratio = b / denom

    mask = (ratio >= ratio_thr) & (b >= compare_min)

    # 背景（base）正規化（全スライス共通）
    lo, hi = np.percentile(a, bg_percentile)
    if hi <= lo:
        hi = lo + 1e-6

    os.makedirs(out_dir, exist_ok=True)

    for z in range(a.shape[0]):
        img = b[z]
        mm  = mask[z]

        img01 = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
        gray = (img01 * 255.0).astype(np.uint8)

        rgb = np.stack([gray, gray, gray], axis=-1).astype(np.float32)

        red = np.zeros_like(rgb)
        red[..., 0] = 255.0
        rgb[mm] = (1.0 - alpha) * rgb[mm] + alpha * red[mm]

        imageio.imwrite(os.path.join(out_dir, f"{prefix}_{z:04d}.{ext}"), rgb.astype(np.uint8))

    print(f"[save_compare_ratio_overlay] ratio_thr={ratio_thr}, base_eps={base_eps}, compare_min={compare_min}, "
          f"mask_count={mask.sum()} / {mask.size}, saved to: {out_dir}")

# 使い方例:
save_compare_ratio_overlay(vol_pred, vol_pred_compare,
                           out_dir="overlay_ratio_2x",
                           ratio_thr=1.2,
                           compare_min=0.05,
                           base_eps=1e-6,
                           order="HWD",
                           alpha=0.6)


[save_compare_ratio_overlay] ratio_thr=1.2, base_eps=1e-06, compare_min=0.05, mask_count=622240 / 16777216, saved to: overlay_ratio_2x


In [72]:
import os
import numpy as np
import torch
import imageio.v2 as imageio
def volume_to_mp4(
    volume: torch.Tensor,            # (D,H,W)
    out_path: str = "volume.mp4",
    fps: int = 24,
    percentile=(1, 99),              # 強い外れ値を無視して見やすくする
):
    assert volume.ndim == 3, "volume must be (D,H,W)"
    vol = volume.detach()
    if vol.is_cuda:
        vol = vol.cpu()
    vol = vol.float().numpy()  # (D,H,W)

    # 画素値スケーリング（全スライス共通のmin/maxで統一）
    lo, hi = np.percentile(vol, percentile)
    if hi <= lo:
        hi = lo + 1e-6

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)

    writer = imageio.get_writer(out_path, fps=fps, codec="libx264", quality=8)
    try:
        for z in range(vol.shape[2]):
            img = vol[:, :, z]
            img = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
            frame = (img * 255.0).astype(np.uint8)      # (H,W)
            frame_rgb = np.stack([frame]*3, axis=-1)    # (H,W,3) にして動画向けに
            writer.append_data(frame_rgb)
    finally:
        writer.close()

In [73]:
volume_to_mp4(vol_pred, "out/ct_slices.mp4", fps=20)

In [81]:
import torch

def tv3d_map(V_dhw: torch.Tensor, eps: float = 1e-6, reduce: str = "isotropic"):
    """
    V_dhw: (D,H,W)
    return: TV map (D,H,W)
    """
    V = V_dhw.float()

    dz = V[1:, :, :] - V[:-1, :, :]   # (D-1,H,W)
    dy = V[:, 1:, :] - V[:, :-1, :]   # (D,H-1,W)
    dx = V[:, :, 1:] - V[:, :, :-1]   # (D,H,W-1)

    # pad to (D,H,W) by duplicating last difference slice
    dz = torch.cat([dz, dz[-1:, :, :]], dim=0)          # (D,H,W)
    dy = torch.cat([dy, dy[:, -1:, :]], dim=1)          # (D,H,W)
    dx = torch.cat([dx, dx[:, :, -1:]], dim=2)          # (D,H,W)

    if reduce == "isotropic":
        tv = torch.sqrt(dx*dx + dy*dy + dz*dz + eps)
    elif reduce == "anisotropic":
        tv = dx.abs() + dy.abs() + dz.abs()
    else:
        raise ValueError("reduce must be 'isotropic' or 'anisotropic'")
    return tv

def detect_highfreq_voxels_by_tv(
    V_dhw: torch.Tensor,
    method="percentile",   # "percentile" or "mad"
    p=95,                  # 上位(100-p)%をHigh-frequencyに
    k=6.0,
    eps=1e-6,
    reduce="isotropic",
    return_response=False,
):
    TV = tv3d_map(V_dhw, eps=eps, reduce=reduce)

    if method == "percentile":
        thr = torch.quantile(TV.flatten(), p/100.0)
        mask = TV >= thr
    elif method == "mad":
        t = TV.flatten()
        med = t.median()
        mad = (t - med).abs().median().clamp_min(1e-12)
        thr = med + k * mad
        mask = TV > thr
    else:
        raise ValueError("method must be 'percentile' or 'mad'")

    if return_response:
        return mask, TV, thr
    return mask


In [82]:
mask = detect_highfreq_voxels_by_tv(
    vol_pred)

print(f"Detected {mask.sum().item()} high-frequency voxels out of {mask.numel()} total voxels.")

Detected 838862 high-frequency voxels out of 16777216 total voxels.


In [83]:
import os
import numpy as np
import torch
import imageio.v2 as imageio

def make_mask_overlay_video(
    volume: torch.Tensor,          # (D,H,W) float/int
    mask: torch.Tensor,            # (D,H,W) bool/0-1
    out_path: str = "overlay.mp4",
    fps: int = 24,
    percentile=(1, 99),            # 背景の正規化（全スライス共通）
    alpha: float = 0.55,           # 赤の重ね具合（0=無し, 1=全赤）
):
    assert volume.ndim == 3 and mask.ndim == 3
    assert volume.shape == mask.shape

    # torch -> numpy
    v = volume.detach()
    m = mask.detach()
    if v.is_cuda: v = v.cpu()
    if m.is_cuda: m = m.cpu()

    v = v.float().numpy()                 # (D,H,W)
    m = (m > 0).numpy().astype(bool)      # (D,H,W) bool

    # 背景のスケール（全スライス共通でチラつき防止）
    lo, hi = np.percentile(v, percentile)
    if hi <= lo:
        hi = lo + 1e-6

    os.makedirs(os.path.dirname(out_path) or ".", exist_ok=True)
    writer = imageio.get_writer(out_path, fps=fps, codec="libx264", quality=8)
    try:
        D = v.shape[0]
        for z in range(D):
            img = v[z]
            mm = m[z]

            # 0..255 グレースケール
            img01 = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
            gray = (img01 * 255.0).astype(np.uint8)  # (H,W)

            # RGB化
            rgb = np.stack([gray, gray, gray], axis=-1).astype(np.float32)  # (H,W,3)

            # 赤オーバーレイ（alpha blending）
            # ここで「赤」は (255,0,0)
            red = np.zeros_like(rgb)
            red[..., 0] = 255.0

            rgb[mm] = (1.0 - alpha) * rgb[mm] + alpha * red[mm]

            writer.append_data(rgb.astype(np.uint8))
    finally:
        writer.close()

# volume: (H,W,D)  ->  (D,H,W)
vol_dhw = vol_pred.permute(2, 0, 1)
mask_dhw   = mask.permute(2, 0, 1)
make_mask_overlay_video(
    vol_dhw,
    mask_dhw,
    out_path="out/ct_slices_highfreq_overlay.mp4",
    fps=20,
    alpha=0.55
)

In [84]:

def save_mask_overlay_frames(
    volume: torch.Tensor,        # (D,H,W) or (H,W,D)
    mask: torch.Tensor,          # volumeと同shape
    out_dir: str = "frames",
    order: str = "DHW",          # "DHW" or "HWD"
    percentile=(1, 99),
    alpha: float = 0.55,
    ext: str = "png",            # "png" / "jpg" など
    prefix: str = "slice",
):
    assert volume.shape == mask.shape, "volumeとmaskのshapeを揃えてください"
    assert volume.ndim == 3

    v = volume.detach()
    m = mask.detach()
    if v.is_cuda: v = v.cpu()
    if m.is_cuda: m = m.cpu()

    v = v.float().numpy()
    m = (m > 0).numpy().astype(bool)

    # (H,W,D) -> (D,H,W) に揃える（処理をシンプルにするため）
    if order.upper() == "HWD":
        v = np.transpose(v, (2, 0, 1))  # (D,H,W)
        m = np.transpose(m, (2, 0, 1))  # (D,H,W)
    elif order.upper() != "DHW":
        raise ValueError('order must be "DHW" or "HWD"')

    D, H, W = v.shape

    # 背景の正規化レンジ（全スライス共通でチラつき防止）
    lo, hi = np.percentile(v, percentile)
    if hi <= lo:
        hi = lo + 1e-6

    os.makedirs(out_dir, exist_ok=True)

    for z in range(D):
        img = v[z]
        mm  = m[z]

        # 0..255 grayscale
        img01 = np.clip((img - lo) / (hi - lo), 0.0, 1.0)
        gray = (img01 * 255.0).astype(np.uint8)

        # RGB化
        rgb = np.stack([gray, gray, gray], axis=-1).astype(np.float32)

        # mask部分だけ赤をalphaでブレンド
        red = np.zeros_like(rgb)
        red[..., 0] = 255.0
        rgb[mm] = (1.0 - alpha) * rgb[mm] + alpha * red[mm]

        frame = rgb.astype(np.uint8)

        fname = os.path.join(out_dir, f"{prefix}_{z:04d}.{ext}")
        imageio.imwrite(fname, frame)

save_mask_overlay_frames(vol_dhw, mask_dhw, out_dir="frames_hp", order="DHW")

TV 3d LOSSの可視化をする